<a href="https://colab.research.google.com/github/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
sys.path.insert(0, r"D:\gitdesktop\Qtrade\qlib")

import qlib
print(qlib.__file__)

import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict
from qlib.tests.data import GetData

/home/shengwang/miniconda3/envs/qlib/lib/python3.13/site-packages/qlib/__init__.py


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
provider_uri = r"/home/shengwang/Documents/qlib/qlib_bin_norm"  # target_dir
qlib.init(provider_uri=provider_uri, region=REG_CN)

[24271:MainThread](2025-08-31 15:16:47,159) INFO - qlib.Initialization - [config.py:451] - default_conf: client.
[24271:MainThread](2025-08-31 15:16:47,160) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[24271:MainThread](2025-08-31 15:16:47,161) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/home/shengwang/Documents/qlib/qlib_bin_norm')}


In [3]:
market = "all"
benchmark = "SH000300"

# train model

In [5]:
###################################
# train model
###################################
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2024-08-31",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2020-12-31",                   # 用来这段时间控制的是 数据预处理时，用来计算“全局参数”的时间范围，比如训练数据中所有特征的均值和方差。
    "instruments": market,
}

task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "device": "cpu",
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",  #158 / 360
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2020-08-31"),           # 训练模型的样本
                "valid": ("2020-09-01", "2021-08-31"),           # 调参或早停用的验证集
                "test": ("2021-09-01", "2024-08-31"),            # 回测阶段评估模型表现的数据集
            },
        },
    },
}

[24271:MainThread](2025-08-31 15:17:23,130) INFO - qlib.timer - [log.py:127] - Time cost: 34.339s | Loading data Done
[24271:MainThread](2025-08-31 15:17:24,481) INFO - qlib.timer - [log.py:127] - Time cost: 0.580s | DropnaLabel Done
[24271:MainThread](2025-08-31 15:17:26,699) INFO - qlib.timer - [log.py:127] - Time cost: 2.215s | CSZScoreNorm Done
[24271:MainThread](2025-08-31 15:17:26,702) INFO - qlib.timer - [log.py:127] - Time cost: 3.572s | fit & process data Done
[24271:MainThread](2025-08-31 15:17:26,703) INFO - qlib.timer - [log.py:127] - Time cost: 37.913s | Init data Done
[24271:MainThread](2025-08-31 15:17:26,786) INFO - qlib.workflow - [exp.py:258] - Experiment 113477453243012254 starts running ...
[24271:MainThread](2025-08-31 15:17:26,810) INFO - qlib.workflow - [recorder.py:345] - Recorder 686cb845bab74751ac2e83ddb47bef43 starts running under Experiment 113477453243012254 ...


数据开始日期： (Timestamp('2008-01-02 00:00:00'), 'SH600000')
数据结束日期： (Timestamp('2024-08-30 00:00:00'), 'SZ301269')
数据的 features 列： ['KMID', 'KLEN', 'KMID2', 'KUP', 'KUP2', 'KLOW', 'KLOW2', 'KSFT', 'KSFT2', 'OPEN0', 'HIGH0', 'LOW0', 'VWAP0', 'ROC5', 'ROC10', 'ROC20', 'ROC30', 'ROC60', 'MA5', 'MA10', 'MA20', 'MA30', 'MA60', 'STD5', 'STD10', 'STD20', 'STD30', 'STD60', 'BETA5', 'BETA10', 'BETA20', 'BETA30', 'BETA60', 'RSQR5', 'RSQR10', 'RSQR20', 'RSQR30', 'RSQR60', 'RESI5', 'RESI10', 'RESI20', 'RESI30', 'RESI60', 'MAX5', 'MAX10', 'MAX20', 'MAX30', 'MAX60', 'MIN5', 'MIN10', 'MIN20', 'MIN30', 'MIN60', 'QTLU5', 'QTLU10', 'QTLU20', 'QTLU30', 'QTLU60', 'QTLD5', 'QTLD10', 'QTLD20', 'QTLD30', 'QTLD60', 'RANK5', 'RANK10', 'RANK20', 'RANK30', 'RANK60', 'RSV5', 'RSV10', 'RSV20', 'RSV30', 'RSV60', 'IMAX5', 'IMAX10', 'IMAX20', 'IMAX30', 'IMAX60', 'IMIN5', 'IMIN10', 'IMIN20', 'IMIN30', 'IMIN60', 'IMXD5', 'IMXD10', 'IMXD20', 'IMXD30', 'IMXD60', 'CORR5', 'CORR10', 'CORR20', 'CORR30', 'CORR60', 'CORD5', 'CORD1

[24271:MainThread](2025-08-31 15:17:41,090) INFO - qlib.timer - [log.py:127] - Time cost: 0.026s | waiting `async_log` Done


Early stopping, best iteration is:
[14]	train's l2: 0.993013	valid's l2: 0.996361


In [1]:
dataset = init_instance_by_config(task["dataset"])


NameError: name 'init_instance_by_config' is not defined

In [ ]:
model = init_instance_by_config(task["model"])
# start exp to train model
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id